# Notebook 3 – Vector Embedding & DBSCAN Clustering

This notebook embeds all extracted YOLO crops of "Die Bombe" with and clusters them for the discovery of the character-prints of the periodical.

##Models & Parameters used
All extracted YOLO crops of "Die Bombe" are embedded with [**laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K**](https://huggingface.co/laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K), a CLIP-model trained on the Data provided via [DataComp](http://www.datacomp.ai/).

##Clustering
The clustering algorithm used in this notebook is **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise), a clustering method which needs two parameters set by the user:
1) the epsilon-radius around each data-point
2) K = the minimum points which need to be found inside the epsilon-radius for it to be considered a corepoint

##Visual Inspection
For the intuitive inspection of the clusters, this notebook writes a html-view for easier access to the predicted clustering.

##Export for Manual Curation
The final clustering result is saved to a CSV which can be used in Notebook-4 for the manual curation of the cluster-groups (if needed) and for the final export.

##Environment
This notebook was created with the help of ChatGPT-5.5 and is supposed to be used in a Google Colab/Google Drive environment.



## 1) Installing the Dependencies and Importing the Libraries
This cell pins certain dependencies necessary for the CLIP embedding to run. The notebook needs a **runtime restart** after running this cell.

In [ ]:
# =====================================================
# Install required packages
# =====================================================

!pip -q uninstall -y numpy scipy torchvision torchaudio open_clip_torch transformers || true

!pip -q install --no-cache-dir --force-reinstall \
    "numpy==2.2.2" \
    "scipy==1.14.1" \
    "pandas==2.2.2" \
    "pillow<12" \
    "torchvision" \
    "open_clip_torch" \
    "pyarrow" \
    "tqdm"

print("Installation successful!")
print("Restart the runtime ONCE and continue with the next cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 252.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 252.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 239.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 247.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 227.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 164.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 272.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 211.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 201.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 159.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Importing libraries
import os
import gc
import time
import re
import math
import json
import platform
import subprocess
import sys

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from html import escape
from urllib.parse import quote
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
import open_clip

from sklearn.cluster import DBSCAN

## 2) Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


## 3) Configurating Paths and Settings
The paths need to be adjusted to the correct directory structureand the path to the metadata-csv-file for the crops must be integrated.

Inside this cell the paths for the embedding- and the clustering-runs can be set, as well as the parameters for the clustering can be changed. Each changed parameter will create a new directory within the "clustering" folder for inspection and reproducability.

In [ ]:
# =====================================================
# 3) Configure paths and settings
# =====================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Masterarbeit_DH/"
    "Pipeline_building-character"
)

CROPS_META_CSV = (
    DRIVE_ROOT
    / "Data/Crops_Bombe/metadata"
    / "Crops_Bombe_metadata.csv"
)


# -----------------------------------------------------
# Embedding run output
# -----------------------------------------------------

RUN_TAG = "datacomp_vitl14_dbscan"

OUT_BASE = (
    DRIVE_ROOT
    / "Embeddings"
    / RUN_TAG
)

OUT_EMB = (
    OUT_BASE
    / "embeddings"
)

OUT_META = (
    OUT_BASE
    / "metadata"
)

OUT_CLUSTER = (
    OUT_BASE
    / "clustering"
)

RUN_META_DIR = (
    OUT_BASE
    / "run_metadata"
)


# -----------------------------------------------------
# OpenCLIP embedding model
# -----------------------------------------------------

OPENCLIP_MODEL_NAME = "ViT-L-14"

OPENCLIP_PRETRAINED = (
    "datacomp_xl_s13b_b90k"
)

MODEL_TAG = (
    "openclip_vitl14_"
    "datacomp_xl_s13b_b90k"
)

EMBEDDING_DIM = 768
BATCH_SIZE = 16
NUM_WORKERS = 0


# -----------------------------------------------------
# Embedding output files
# -----------------------------------------------------

INDEX_PATH = (
    OUT_EMB
    / f"{MODEL_TAG}_index.parquet"
)

EMB_MEMMAP_PATH = (
    OUT_EMB
    / f"{MODEL_TAG}_embeddings.memmap"
)

DONE_MASK_PATH = (
    OUT_EMB
    / f"{MODEL_TAG}_done_mask.npy"
)

EMB_NPY_PATH = (
    OUT_EMB
    / f"{MODEL_TAG}_embeddings.npy"
)


# -----------------------------------------------------
# DBSCAN clustering
# -----------------------------------------------------

DBSCAN_EPS = 0.08
DBSCAN_MIN_SAMPLES = 3
DBSCAN_METRIC = "cosine"


# -----------------------------------------------------
# Parameter-specific clustering variant directory
# -----------------------------------------------------

def format_decimal_parameter(
    value,
    decimal_places=2,
):
    """
    Convert decimal parameters into filename-safe strings.

    Examples:
        0.12 -> 012
        0.10 -> 010
        0.08 -> 008
    """
    return (
        f"{value:.{decimal_places}f}"
        .replace(".", "")
        .replace("-", "neg")
    )


DBSCAN_EPS_TAG = format_decimal_parameter(
    DBSCAN_EPS,
    decimal_places=2,
)

CLUSTERING_VARIANT_NAME = (
    f"dbscan_eps{DBSCAN_EPS_TAG}"
    f"_min{DBSCAN_MIN_SAMPLES}"
    f"_{DBSCAN_METRIC}"
)

CLUSTERING_VARIANT_DIR = (
    OUT_CLUSTER
    / CLUSTERING_VARIANT_NAME
)


# -----------------------------------------------------
# Create output directories
# -----------------------------------------------------

for path in [
    OUT_BASE,
    OUT_EMB,
    OUT_META,
    OUT_CLUSTER,
    RUN_META_DIR,
    CLUSTERING_VARIANT_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# -----------------------------------------------------
# Validate important paths
# -----------------------------------------------------

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        "Pipeline root does not exist:\n"
        f"{DRIVE_ROOT}"
    )

if not CROPS_META_CSV.is_file():
    raise FileNotFoundError(
        "Crop metadata CSV does not exist:\n"
        f"{CROPS_META_CSV}"
    )


# -----------------------------------------------------
# Display resolved configuration
# -----------------------------------------------------

print("=" * 70)
print("PATH AND RUN CONFIGURATION")
print("=" * 70)

print("Pipeline root:")
print(DRIVE_ROOT)

print("\nCrop metadata:")
print(CROPS_META_CSV)

print("\nEmbedding run directory:")
print(OUT_BASE)

print("\nEmbedding output directory:")
print(OUT_EMB)

print("\nMetadata output directory:")
print(OUT_META)

print("\nClustering output directory:")
print(OUT_CLUSTER)

print("\nActive clustering variant:")
print(CLUSTERING_VARIANT_DIR)

print("\nEmbedding index:")
print(INDEX_PATH)

print("\nEmbedding memmap:")
print(EMB_MEMMAP_PATH)

print("\nDone mask:")
print(DONE_MASK_PATH)

print("=" * 70)

PATH AND RUN CONFIGURATION
Pipeline root:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character

Crop metadata:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Crops_Bombe/metadata/Crops_Bombe_metadata.csv

Embedding run directory:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan

Embedding output directory:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/embeddings

Metadata output directory:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/metadata

Clustering output directory:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering

Active clustering variant:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine

Embedding index:
/content/drive/MyDrive/Ma

##4) Creation of Run-Metadata
Here the metadata of each run can be saved to the folder. Only needs to be run once, except if important changes were made to the settings.

In [ ]:
# === Reproducibility / run logging ===

# OUT_BASE.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# RUN_META_DIR = (
#     OUT_BASE
#     / "run_metadata"
# )

# RUN_META_DIR.mkdir(
#     parents=True,
#     exist_ok=True,
# )


# # -----------------------------------------------------
# # Stable run configuration
# # -----------------------------------------------------

# run_config = {
#     "timestamp_utc": (
#         datetime.now(
#             timezone.utc
#         ).isoformat()
#     ),

#     "run_tag": RUN_TAG,

#     "drive_root": str(
#         DRIVE_ROOT
#     ),

#     "crops_meta_csv": str(
#         CROPS_META_CSV
#     ),

#     "embedding_model": {
#         "library": "open_clip",
#         "model_name": OPENCLIP_MODEL_NAME,
#         "pretrained": OPENCLIP_PRETRAINED,
#         "embedding_dim": EMBEDDING_DIM,
#         "batch_size": BATCH_SIZE,
#         "num_workers": NUM_WORKERS,
#         "l2_normalized": True,
#     },

#     # The detailed DBSCAN parameters are documented
#     # separately inside each clustering-variant folder.
#     "clustering_algorithm": "DBSCAN",

#     "paths": {
#         "index_path": str(
#             INDEX_PATH
#         ),

#         "embedding_memmap_path": str(
#             EMB_MEMMAP_PATH
#         ),

#         "done_mask_path": str(
#             DONE_MASK_PATH
#         ),

#         "embedding_npy_path": str(
#             EMB_NPY_PATH
#         ),

#         "metadata_output_dir": str(
#             OUT_META
#         ),

#         "clustering_output_dir": str(
#             OUT_CLUSTER
#         ),
#     },
# }


# with open(
#     RUN_META_DIR
#     / "run_config.json",
#     "w",
#     encoding="utf-8",
# ) as file:
#     json.dump(
#         run_config,
#         file,
#         indent=2,
#         ensure_ascii=False,
#     )


# # -----------------------------------------------------
# # Runtime environment information
# # -----------------------------------------------------

# env = {
#     "python": sys.version,
#     "platform": platform.platform(),
#     "numpy": np.__version__,
#     "pandas": pd.__version__,
# }


# try:
#     env.update(
#         {
#             "torch": torch.__version__,
#             "cuda_available": (
#                 torch.cuda.is_available()
#             ),
#             "cuda_version": (
#                 torch.version.cuda
#             ),
#             "cudnn_version": (
#                 torch.backends.cudnn.version()
#                 if torch.cuda.is_available()
#                 else None
#             ),
#             "gpu_name": (
#                 torch.cuda.get_device_name(0)
#                 if torch.cuda.is_available()
#                 else None
#             ),
#         }
#     )

# except Exception as error:
#     env["torch_error"] = repr(
#         error
#     )


# try:
#     env["open_clip"] = (
#         open_clip.__version__
#     )

# except Exception:
#     env["open_clip"] = (
#         "version unavailable"
#     )


# try:
#     import sklearn

#     env["scikit_learn"] = (
#         sklearn.__version__
#     )

# except Exception as error:
#     env["scikit_learn_error"] = repr(
#         error
#     )


# with open(
#     RUN_META_DIR
#     / "environment.json",
#     "w",
#     encoding="utf-8",
# ) as file:
#     json.dump(
#         env,
#         file,
#         indent=2,
#         ensure_ascii=False,
#     )


# # -----------------------------------------------------
# # Complete package list
# # -----------------------------------------------------

# try:
#     freeze = subprocess.check_output(
#         [
#             sys.executable,
#             "-m",
#             "pip",
#             "freeze",
#         ],
#         text=True,
#     )

#     (
#         RUN_META_DIR
#         / "pip_freeze.txt"
#     ).write_text(
#         freeze,
#         encoding="utf-8",
#     )

# except Exception as error:
#     (
#         RUN_META_DIR
#         / "pip_freeze.txt"
#     ).write_text(
#         (
#             "pip freeze failed: "
#             f"{repr(error)}"
#         ),
#         encoding="utf-8",
#     )


# print(
#     "Saved stable run metadata to:",
#     RUN_META_DIR,
# )

Saved stable run metadata to: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/run_metadata


## 5) Loading Crop-Metadata
The required columns of the crop metadata can be chosen here.

In [ ]:
# =====================================================
# Load crop metadata
# =====================================================

df_crops = pd.read_csv(CROPS_META_CSV)

# Metadata required by the embedding pipeline
required_cols = [
    "crop_id",
    "page_id",
    "category",
    "local_crop_path",
]

missing = [
    c
    for c in required_cols
    if c not in df_crops.columns
]

if missing:
    raise ValueError(
        f"Missing columns in crop metadata: {missing}\n"
        f"Available columns: {list(df_crops.columns)}"
    )


# -----------------------------------------------------
# Keep the relative path for HTML reports
# -----------------------------------------------------

df_crops["local_crop_path"] = (
    df_crops["local_crop_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.strip()
)


# -----------------------------------------------------
# Create an absolute path for embedding
# -----------------------------------------------------

df_crops["absolute_crop_path"] = (
    DRIVE_ROOT
    / df_crops["local_crop_path"].map(Path)
).astype(str)


df_crops = df_crops.reset_index(drop=True)


print(f"Crops: {len(df_crops):,}")
print()

print("Example relative path:")
print(df_crops.loc[0, "local_crop_path"])

print()

print("Example absolute path:")
print(df_crops.loc[0, "absolute_crop_path"])

display(df_crops.head(3))

Crops: 36,257

Example relative path:
Data/Crops_Bombe/Illustration/bom-crop_00000001.jpg

Example absolute path:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Crops_Bombe/Illustration/bom-crop_00000001.jpg


,crop_id,page_id,category,category_number,bbox_xyxy,bbox_xywh,confidence,iiif_crop_url,iiif_page_url,local_crop_path,model_weights,imgsz,conf_threshold,anno_id,absolute_crop_path
0,bom-crop_00000001,bom18710108_001,Illustration,4,"(59,1466,2133,3238)","(59,1466,2074,1772)",0.253965,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,Data/Crops_Bombe/Illustration/bom-crop_0000000...,Model/runs/yolov8m_final-pipeline_run-001/weig...,1280,0.25,bom18710108,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...
1,bom-crop_00000002,bom18710108_003,Illustration,4,"(294,299,2134,1741)","(294,299,1840,1442)",0.679213,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,Data/Crops_Bombe/Illustration/bom-crop_0000000...,Model/runs/yolov8m_final-pipeline_run-001/weig...,1280,0.25,bom18710108,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...
2,bom-crop_00000003,bom18710108_004,Illustration,4,"(1322,878,2789,1472)","(1322,878,1467,594)",0.672864,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,https://api.onb.ac.at/iiif/image/v3/11D6BA1C/u...,Data/Crops_Bombe/Illustration/bom-crop_0000000...,Model/runs/yolov8m_final-pipeline_run-001/weig...,1280,0.25,bom18710108,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...


## 6) Auxiliary Functions for CLIP

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load OpenCLIP ViT-L/14 with DataComp weights
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=OPENCLIP_MODEL_NAME,
    pretrained=OPENCLIP_PRETRAINED,
    device=device,
)

model.eval()


@torch.inference_mode()
def embed_images_clip(paths, batch_size=BATCH_SIZE):
    """
    Return L2-normalized OpenCLIP image embeddings as float32.

    Output shape:
        (number_of_images, 768)
    """
    all_emb = []

    for i in tqdm(
        range(0, len(paths), batch_size),
        desc="Embedding",
    ):
        batch_paths = paths[i:i + batch_size]

        images = []

        for path in batch_paths:
            with Image.open(path) as image:
                image = image.convert("RGB")
                images.append(preprocess(image))

        image_tensor = torch.stack(images).to(device)

        features = model.encode_image(image_tensor)

        # Convert to float32 before normalization.
        features = features.float()

        # L2 normalization
        features = features / features.norm(
            dim=-1,
            keepdim=True,
        )

        all_emb.append(
            features.cpu().numpy().astype("float32")
        )

        del images, image_tensor, features

        if device == "cuda":
            torch.cuda.empty_cache()

    embeddings = np.vstack(all_emb).astype("float32")

    if embeddings.shape[1] != EMBEDDING_DIM:
        raise ValueError(
            f"Expected embedding dimension {EMBEDDING_DIM}, "
            f"but received {embeddings.shape[1]}."
        )

    return embeddings

Device: cpu


## 7) Compute/Load CLIP-embeddings
This cell will compute the CLIP-embeddings for the crops or load them if they already have been computed. It is interruption-safe and will restart at the last computed embedding, in the case of an interruption during the embedding process.

In [ ]:
# ---- output files ----
OUT_EMB.mkdir(parents=True, exist_ok=True)

index_path = INDEX_PATH
emb_mm_path = EMB_MEMMAP_PATH
mask_path = DONE_MASK_PATH


# ---- create/load stable df_index ----

index_columns = [
    "crop_id",
    "page_id",
    "category",
    "local_crop_path",
]

if index_path.exists():
    df_index = pd.read_parquet(index_path)

    missing_index_cols = [
        column
        for column in index_columns
        if column not in df_index.columns
    ]

    if missing_index_cols:
        raise ValueError(
            "The existing embedding index has an outdated structure.\n"
            f"Missing columns: {missing_index_cols}\n"
            f"Index path: {index_path}\n"
            "Delete the existing index and its associated embedding "
            "cache files, or use a new RUN_TAG."
        )

    print(
        f"Loaded existing df_index: "
        f"{len(df_index):,} rows"
    )

else:
    df_index = df_crops[
        index_columns
    ].copy()

    df_index.to_parquet(
        index_path,
        index=False,
    )

    print(
        f"Saved new df_index: {len(df_index):,} rows "
        f"-> {index_path}"
    )


# Normalize relative paths without converting them to absolute paths.
df_index["local_crop_path"] = (
    df_index["local_crop_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.strip()
)


# Create absolute paths only in runtime memory.
df_index["absolute_crop_path"] = (
    DRIVE_ROOT
    / df_index["local_crop_path"].map(Path)
).astype(str)


paths = (
    df_index["absolute_crop_path"]
    .astype(str)
    .tolist()
)

N = len(paths)


# ---- embedding dimensionality ----
# OpenCLIP ViT-L/14 DataComp outputs 768-dimensional embeddings.
D = EMBEDDING_DIM

if D != 768:
    raise ValueError(
        f"Expected EMBEDDING_DIM=768 for "
        f"{OPENCLIP_MODEL_NAME}, but received {D}."
    )


# ---- open/create memmap ----
expected_size_bytes = N * D * np.dtype("float32").itemsize

if emb_mm_path.exists():
    actual_size_bytes = emb_mm_path.stat().st_size

    if actual_size_bytes != expected_size_bytes:
        raise ValueError(
            "Existing embedding memmap has an incompatible size.\n"
            f"Expected: {expected_size_bytes:,} bytes "
            f"for shape ({N}, {D})\n"
            f"Found:    {actual_size_bytes:,} bytes\n"
            "Use a new RUN_TAG or delete the incompatible cache files."
        )

    mode = "r+"

else:
    mode = "w+"


X_mm = np.memmap(
    emb_mm_path,
    dtype="float32",
    mode=mode,
    shape=(N, D),
)


# ---- load/create done mask ----
if mask_path.exists():
    done = np.load(mask_path)

    if done.shape != (N,):
        raise ValueError(
            f"done_mask shape {done.shape} != expected shape ({N},).\n"
            "This RUN_TAG's df_index changed. "
            "Use a new RUN_TAG for a new dataset, sample, or row order."
        )

    done = done.astype(bool, copy=False)

else:
    done = np.zeros(N, dtype=bool)


print(
    f"Embedding progress: "
    f"{int(done.sum()):,}/{N:,} done"
)


# ---- configure this session's work chunk ----

CHUNK_SIZE = None

# Number of embeddings written before the memmap and done mask are checkpointed.
SAVE_EVERY = 500


remaining = np.where(~done)[0]

if remaining.size == 0:
    print("All embeddings already computed.")

else:
    todo = (
        remaining
        if CHUNK_SIZE is None
        else remaining[:CHUNK_SIZE]
    )

    print(
        f"Embedding {len(todo):,} crops with "
        f"OpenCLIP {OPENCLIP_MODEL_NAME} "
        f"({OPENCLIP_PRETRAINED})..."
    )

    for start in tqdm(
        range(0, len(todo), SAVE_EVERY),
        desc="Embedding chunks",
    ):
        batch_inds = todo[start:start + SAVE_EVERY]

        batch_paths = [
            paths[index]
            for index in batch_inds
        ]

        X_chunk = embed_images_clip(
            batch_paths,
            batch_size=BATCH_SIZE,
        )

        expected_chunk_shape = (
            len(batch_inds),
            D,
        )

        if X_chunk.shape != expected_chunk_shape:
            raise ValueError(
                f"Unexpected embedding shape: {X_chunk.shape}. "
                f"Expected: {expected_chunk_shape}."
            )

        X_mm[batch_inds, :] = X_chunk.astype(
            "float32",
            copy=False,
        )

        X_mm.flush()

        done[batch_inds] = True
        np.save(mask_path, done)

    print(
        f"Checkpoint saved: "
        f"{int(done.sum()):,}/{N:,} done"
    )


# ---- provide X for downstream steps only if complete ----
if done.all():
    # np.asarray returns a view backed by the memmap.
    # This does not necessarily copy the full matrix into RAM.
    X = np.asarray(X_mm)

    print(
        "Embeddings complete:",
        X.shape,
        X.dtype,
    )

else:
    X = None

    print(
        "Embeddings incomplete. Re-run this cell in the next "
        "session to continue. Downstream clustering cells "
        "require the embedding matrix to be complete."
    )

Loaded existing df_index: 36,257 rows
Embedding progress: 36,257/36,257 done
All embeddings already computed.
Embeddings complete: (36257, 768) float32


##8) DBSCAN Clustering

The [sklearn python library](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) is used to implement the DBSCAN clustering - parameters need to be adjusted to match the dataset.


In [ ]:
# =====================================================
# 7) DBSCAN clustering
# =====================================================

eps = DBSCAN_EPS
min_samples = DBSCAN_MIN_SAMPLES
metric = DBSCAN_METRIC


# -----------------------------------------------------
# Validate embedding data
# -----------------------------------------------------

if X is None:
    raise RuntimeError(
        "The embedding matrix is unavailable. "
        "Finish or load the embeddings before running DBSCAN."
    )

if not done.all():
    raise RuntimeError(
        f"Only {int(done.sum()):,} of "
        f"{len(done):,} embeddings are complete."
    )

if len(X) != len(df_index):
    raise ValueError(
        f"Embedding rows ({len(X):,}) do not match "
        f"index rows ({len(df_index):,})."
    )

if not np.isfinite(X).all():
    raise ValueError(
        "The embedding matrix contains NaN or infinite values."
    )


# -----------------------------------------------------
# Run DBSCAN
# -----------------------------------------------------

print("=" * 70)
print("STARTING DBSCAN CLUSTERING")
print("=" * 70)
print(f"Images:       {len(X):,}")
print(f"Dimensions:   {X.shape[1]:,}")
print(f"eps:          {eps}")
print(f"MinPts:       {min_samples}")
print(f"Metric:       {metric}")
print()

start_time = time.time()

dbscan_model = DBSCAN(
    eps=eps,
    min_samples=min_samples,
    metric=metric,
    algorithm="brute",
    n_jobs=-1,
)

cluster_labels = dbscan_model.fit_predict(X)
clustering_time = time.time() - start_time


# -----------------------------------------------------
# Create image-level clustering metadata
# -----------------------------------------------------

metadata_columns = [
    "crop_id",
    "page_id",
    "category",
    "local_crop_path",
]

missing_columns = [
    column
    for column in metadata_columns
    if column not in df_index.columns
]

if missing_columns:
    raise ValueError(
        "The embedding index is missing required columns: "
        f"{missing_columns}"
    )

clustering_metadata = (
    df_index[metadata_columns]
    .copy()
    .reset_index(drop=True)
)

clustering_metadata.insert(
    0,
    "embedding_index",
    np.arange(len(clustering_metadata)),
)

clustering_metadata["cluster_id"] = cluster_labels
clustering_metadata["is_noise"] = (
    clustering_metadata["cluster_id"] == -1
)


# -----------------------------------------------------
# Add cluster sizes
# -----------------------------------------------------

cluster_sizes = (
    clustering_metadata.loc[
        ~clustering_metadata["is_noise"],
        "cluster_id",
    ]
    .value_counts()
    .sort_index()
)

clustering_metadata["cluster_size"] = (
    clustering_metadata["cluster_id"]
    .map(cluster_sizes)
    .fillna(1)
    .astype(int)
)


# -----------------------------------------------------
# Calculate clustering statistics
# -----------------------------------------------------

noise_mask = cluster_labels == -1

number_of_images = len(cluster_labels)
number_of_noise_images = int(noise_mask.sum())
number_of_clustered_images = int((~noise_mask).sum())

unique_cluster_ids = np.unique(
    cluster_labels[~noise_mask]
)

number_of_clusters = len(unique_cluster_ids)

noise_percentage = (
    number_of_noise_images
    / number_of_images
    * 100
)

clustered_percentage = (
    number_of_clustered_images
    / number_of_images
    * 100
)


print("=" * 70)
print("DBSCAN CLUSTERING COMPLETE")
print("=" * 70)
print(
    f"Runtime:            "
    f"{clustering_time / 60:.2f} minutes"
)
print(
    f"Clusters:           "
    f"{number_of_clusters:,}"
)
print(
    f"Clustered images:   "
    f"{number_of_clustered_images:,} "
    f"({clustered_percentage:.2f}%)"
)
print(
    f"Noise images:       "
    f"{number_of_noise_images:,} "
    f"({noise_percentage:.2f}%)"
)

if number_of_clusters > 0:
    print(
        f"Smallest cluster:   "
        f"{int(cluster_sizes.min()):,} images"
    )
    print(
        f"Largest cluster:    "
        f"{int(cluster_sizes.max()):,} images"
    )
    print(
        f"Mean cluster size:  "
        f"{cluster_sizes.mean():.2f} images"
    )
    print(
        f"Median cluster size: "
        f"{cluster_sizes.median():.2f} images"
    )

print("=" * 70)


# -----------------------------------------------------
# Create cluster-level metadata
# -----------------------------------------------------

cluster_level_metadata = (
    clustering_metadata.loc[
        ~clustering_metadata["is_noise"]
    ]
    .groupby("cluster_id")
    .agg(
        cluster_size=("embedding_index", "size"),
    )
    .reset_index()
    .sort_values(
        by=[
            "cluster_size",
            "cluster_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# -----------------------------------------------------
# Save clustering outputs
# -----------------------------------------------------

IMAGE_CLUSTERING_PATH = (
    CLUSTERING_VARIANT_DIR
    / "images.parquet"
)

CLUSTER_LEVEL_PATH = (
    CLUSTERING_VARIANT_DIR
    / "clusters.parquet"
)

CLUSTER_SUMMARY_PATH = (
    CLUSTERING_VARIANT_DIR
    / "summary.json"
)

clustering_metadata.to_parquet(
    IMAGE_CLUSTERING_PATH,
    index=False,
)

cluster_level_metadata.to_parquet(
    CLUSTER_LEVEL_PATH,
    index=False,
)

cluster_summary = {
    "algorithm": "DBSCAN",
    "embedding_model": MODEL_TAG,
    "parameters": {
        "eps": eps,
        "min_samples": min_samples,
        "metric": metric,
    },
    "number_of_images": int(number_of_images),
    "number_of_clusters": int(number_of_clusters),
    "number_of_clustered_images": (
        number_of_clustered_images
    ),
    "number_of_noise_images": number_of_noise_images,
    "clustered_percentage": clustered_percentage,
    "noise_percentage": noise_percentage,
    "runtime_seconds": clustering_time,
    "smallest_cluster": (
        int(cluster_sizes.min())
        if number_of_clusters > 0
        else None
    ),
    "largest_cluster": (
        int(cluster_sizes.max())
        if number_of_clusters > 0
        else None
    ),
    "mean_cluster_size": (
        float(cluster_sizes.mean())
        if number_of_clusters > 0
        else None
    ),
    "median_cluster_size": (
        float(cluster_sizes.median())
        if number_of_clusters > 0
        else None
    ),
}

with open(
    CLUSTER_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cluster_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\nSaved image-level clustering metadata:")
print(IMAGE_CLUSTERING_PATH)

print("\nSaved cluster-level metadata:")
print(CLUSTER_LEVEL_PATH)

print("\nSaved clustering summary:")
print(CLUSTER_SUMMARY_PATH)

display(clustering_metadata.head())

STARTING DBSCAN CLUSTERING
Images:       36,257
Dimensions:   768
eps:          0.08
MinPts:       3
Metric:       cosine

DBSCAN CLUSTERING COMPLETE
Runtime:            1.36 minutes
Clusters:           818
Clustered images:   18,011 (49.68%)
Noise images:       18,246 (50.32%)
Smallest cluster:   3 images
Largest cluster:    3,791 images
Mean cluster size:  22.02 images
Median cluster size: 5.00 images

Saved image-level clustering metadata:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/images.parquet

Saved cluster-level metadata:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/clusters.parquet

Saved clustering summary:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/summary.json


,embedding_index,crop_id,page_id,category,local_crop_path,cluster_id,is_noise,cluster_size
0,0,bom-crop_00000001,bom18710108_001,Illustration,Data/Crops_Bombe/Illustration/bom-crop_0000000...,-1,True,1
1,1,bom-crop_00000002,bom18710108_003,Illustration,Data/Crops_Bombe/Illustration/bom-crop_0000000...,-1,True,1
2,2,bom-crop_00000003,bom18710108_004,Illustration,Data/Crops_Bombe/Illustration/bom-crop_0000000...,-1,True,1
3,3,bom-crop_00000004,bom18710108_004,Illustration,Data/Crops_Bombe/Illustration/bom-crop_0000000...,-1,True,1
4,4,bom-crop_00000005,bom18710108_004,Illustration,Data/Crops_Bombe/Illustration/bom-crop_0000000...,-1,True,1


##9) HTML Report for Cluster Inspection

This cell builds a simple HTML-view of the clusters created by the DBSCAN algorithm implemented on the Vector-Embeddings. It should be used for a first visual inspection and be a help for tweaking the parameters to obtain a satisfying clustering result.

In [ ]:
# =====================================================
# Create lightweight HTML cluster inspection report
# =====================================================

HTML_REPORT_PATH = (
    CLUSTERING_VARIANT_DIR
    / "cluster_inspection.html"
)

# Column containing paths relative to the pipeline root.
IMAGE_PATH_COLUMN = "local_crop_path"

# must be set to match the needed directory-structure
IMAGE_URL_PREFIX = "../../../../"

# DBSCAN noise has cluster_id == -1.
INCLUDE_NOISE = False

# Set to an integer to limit the number shown per cluster, set to None to make all images available in every cluster.
MAX_IMAGES_PER_CLUSTER = 100

THUMBNAIL_SIZE = 180

INITIAL_OPEN_CLUSTERS = 0

PROGRESS_EVERY_N_CLUSTERS = 100

# -----------------------------------------------------
# Validate required data
# -----------------------------------------------------

required_columns = {
    "cluster_id",
    "embedding_index",
    IMAGE_PATH_COLUMN,
}

missing_columns = (
    required_columns
    - set(clustering_metadata.columns)
)

if missing_columns:
    raise ValueError(
        "The clustering metadata is missing required columns: "
        f"{sorted(missing_columns)}\n"
        f"Available columns: {list(clustering_metadata.columns)}"
    )

print("=" * 70)
print("CREATING LIGHTWEIGHT HTML CLUSTER REPORT")
print("=" * 70)
print("HTML report:", HTML_REPORT_PATH)
print("Image-path column:", IMAGE_PATH_COLUMN)
print("Image URL prefix:", repr(IMAGE_URL_PREFIX))
print("Include noise:", INCLUDE_NOISE)
print(
    "Maximum images per cluster:",
    MAX_IMAGES_PER_CLUSTER,
)

# -----------------------------------------------------
# Detect a useful filename column
# -----------------------------------------------------

possible_filename_columns = [
    "filename",
    "file_name",
    "image_filename",
    "crop_filename",
    "crop_id",
]

filename_column = next(
    (
        column
        for column in possible_filename_columns
        if column in clustering_metadata.columns
    ),
    None,
)

if filename_column is not None:
    print("Filename label column:", filename_column)
else:
    print("Filename labels will be extracted from local_crop_path.")


# -----------------------------------------------------
# Convert stored crop path to browser URL
# -----------------------------------------------------

def crop_path_to_browser_url(
    crop_path,
    url_prefix,
):
    """
    Combine IMAGE_URL_PREFIX with a crop path stored relative
    to the pipeline root.

    The path is converted to URL-compatible forward slashes.
    Spaces and other special characters are URL-encoded while
    path separators and common URL characters remain intact.
    """
    crop_path = str(crop_path).strip()

    if not crop_path:
        return None

    # Browser URLs require forward slashes.
    crop_path = crop_path.replace("\\", "/")

    # Prevent an accidental duplicate slash at the join point.
    prefix = str(url_prefix).strip()

    if prefix:
        browser_path = (
            prefix.rstrip("/")
            + "/"
            + crop_path.lstrip("/")
        )
    else:
        browser_path = crop_path

    # Encode spaces and special characters but retain URL syntax.
    browser_path = quote(
        browser_path,
        safe="/:@?&=#%+-._~",
    )

    return browser_path


# -----------------------------------------------------
# Prepare report data
# -----------------------------------------------------

report_data = clustering_metadata.copy()

if not INCLUDE_NOISE:
    report_data = report_data.loc[
        report_data["cluster_id"] != -1
    ].copy()

report_data = report_data.sort_values(
    [
        "cluster_id",
        "embedding_index",
    ]
)

cluster_groups = {
    int(cluster_id): group
    for cluster_id, group in report_data.groupby(
        "cluster_id",
        sort=False,
    )
}

# Display largest clusters first.
ordered_cluster_ids = sorted(
    cluster_groups,
    key=lambda cluster_id: len(
        cluster_groups[cluster_id]
    ),
    reverse=True,
)

number_of_report_clusters = len(
    ordered_cluster_ids
)

number_of_clustered_images = len(
    report_data
)

if MAX_IMAGES_PER_CLUSTER is None:
    number_of_available_images = (
        number_of_clustered_images
    )
else:
    number_of_available_images = sum(
        min(
            len(cluster_groups[cluster_id]),
            MAX_IMAGES_PER_CLUSTER,
        )
        for cluster_id in ordered_cluster_ids
    )

print(
    "Report clusters:",
    f"{number_of_report_clusters:,}",
)

print(
    "Clustered images:",
    f"{number_of_clustered_images:,}",
)

print(
    "Images available in report:",
    f"{number_of_available_images:,}",
)


# -----------------------------------------------------
# Build compact cluster data
# -----------------------------------------------------

cluster_records = []
cluster_headers = []

start_time = time.perf_counter()


for cluster_position, cluster_id in enumerate(
    ordered_cluster_ids,
    start=1,
):
    cluster_rows = cluster_groups[cluster_id]
    cluster_size = len(cluster_rows)

    if MAX_IMAGES_PER_CLUSTER is None:
        displayed_rows = cluster_rows
    else:
        displayed_rows = cluster_rows.head(
            MAX_IMAGES_PER_CLUSTER
        )

    image_records = []

    for row in displayed_rows.itertuples(
        index=False
    ):
        row_data = row._asdict()

        stored_crop_path = str(
            row_data[IMAGE_PATH_COLUMN]
        ).strip()

        image_url = crop_path_to_browser_url(
            crop_path=stored_crop_path,
            url_prefix=IMAGE_URL_PREFIX,
        )

        if filename_column is not None:
            filename = str(
                row_data[filename_column]
            )
        else:
            filename = Path(
                stored_crop_path
            ).name

        image_records.append(
            {
                "filename": filename,
                "embedding_index": int(
                    row_data["embedding_index"]
                ),
                "stored_path": stored_crop_path,
                "image_url": image_url,
            }
        )

    hidden_count = (
        cluster_size
        - len(displayed_rows)
    )

    cluster_records.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_size": int(cluster_size),
            "hidden_count": int(hidden_count),
            "images": image_records,
        }
    )

    open_attribute = (
        "open"
        if cluster_position <= INITIAL_OPEN_CLUSTERS
        else ""
    )

    cluster_headers.append(
        f"""
        <details
            class="cluster"
            data-cluster-position="{cluster_position - 1}"
            {open_attribute}
        >
            <summary>
                <span class="cluster-title">
                    Cluster {int(cluster_id)}
                </span>

                <span class="cluster-size">
                    {int(cluster_size):,} images
                </span>
            </summary>

            <div class="cluster-content">
                <p class="loading-message">
                    Open this cluster to load its images.
                </p>
            </div>
        </details>
        """
    )

    if (
        cluster_position
        % PROGRESS_EVERY_N_CLUSTERS
        == 0
        or cluster_position
        == number_of_report_clusters
    ):
        print(
            f"Prepared "
            f"{cluster_position:,}/"
            f"{number_of_report_clusters:,} clusters"
        )


# Compact JSON reduces the HTML size.
cluster_data_json = json.dumps(
    cluster_records,
    ensure_ascii=False,
    separators=(",", ":"),
)


# -----------------------------------------------------
# Build complete HTML document
# -----------------------------------------------------

html_document = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>
        DBSCAN Cluster Inspection
    </title>

    <style>
        :root {{
            color-scheme: light;
            font-family:
                Arial,
                Helvetica,
                sans-serif;
        }}

        * {{
            box-sizing: border-box;
        }}

        body {{
            margin: 0;
            padding: 24px;
            background: #f3f4f6;
            color: #111827;
        }}

        .page {{
            max-width: 1800px;
            margin: 0 auto;
        }}

        h1 {{
            margin: 0 0 8px;
        }}

        .subtitle {{
            margin-top: 0;
            color: #4b5563;
        }}

        .path-note {{
            margin: 12px 0 20px;
            padding: 10px 12px;
            border: 1px solid #d1d5db;
            border-radius: 7px;
            background: white;
            color: #4b5563;
            font-size: 13px;
        }}

        code {{
            overflow-wrap: anywhere;
        }}

        .summary {{
            display: flex;
            flex-wrap: wrap;
            gap: 12px;
            margin: 24px 0;
        }}

        .summary-card {{
            min-width: 170px;
            padding: 14px 18px;
            border: 1px solid #d1d5db;
            border-radius: 9px;
            background: white;
        }}

        .summary-label {{
            display: block;
            margin-bottom: 5px;
            color: #6b7280;
            font-size: 13px;
        }}

        .summary-value {{
            font-size: 21px;
            font-weight: bold;
        }}

        .controls {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            margin-bottom: 20px;
        }}

        button {{
            padding: 9px 14px;
            border: 1px solid #9ca3af;
            border-radius: 7px;
            background: white;
            cursor: pointer;
        }}

        button:hover {{
            background: #f9fafb;
        }}

        .cluster {{
            margin-bottom: 12px;
            border: 1px solid #d1d5db;
            border-radius: 9px;
            background: white;
            overflow: hidden;
        }}

        .cluster summary {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 14px 18px;
            background: #e5e7eb;
            cursor: pointer;
            font-weight: bold;
        }}

        .cluster summary:hover {{
            background: #dbe0e7;
        }}

        .cluster-title {{
            font-size: 17px;
        }}

        .cluster-size {{
            color: #4b5563;
            font-size: 14px;
        }}

        .cluster-content {{
            padding: 16px;
        }}

        .loading-message {{
            margin: 0;
            color: #6b7280;
        }}

        .image-grid {{
            display: grid;
            grid-template-columns:
                repeat(
                    auto-fill,
                    minmax({THUMBNAIL_SIZE}px, 1fr)
                );
            gap: 14px;
        }}

        .image-card {{
            min-width: 0;
            padding: 9px;
            border: 1px solid #e5e7eb;
            border-radius: 8px;
            background: #fafafa;
        }}

        .image-link {{
            display: block;
            text-decoration: none;
        }}

        .image-card img {{
            display: block;
            width: 100%;
            height: {THUMBNAIL_SIZE}px;
            object-fit: contain;
            border-radius: 5px;
            background: white;
        }}

        .missing-image {{
            display: flex;
            align-items: center;
            justify-content: center;
            width: 100%;
            height: {THUMBNAIL_SIZE}px;
            border-radius: 5px;
            background: #e5e7eb;
            color: #6b7280;
            text-align: center;
        }}

        .image-info {{
            margin-top: 8px;
        }}

        .filename {{
            overflow-wrap: anywhere;
            font-size: 12px;
            font-weight: bold;
        }}

        .embedding-index {{
            margin-top: 4px;
            color: #6b7280;
            font-size: 11px;
        }}

        .stored-path {{
            margin-top: 4px;
            overflow-wrap: anywhere;
            color: #6b7280;
            font-size: 10px;
        }}

        .truncation-note {{
            margin-bottom: 0;
            color: #6b7280;
            font-style: italic;
        }}

        @media (max-width: 600px) {{
            body {{
                padding: 12px;
            }}

            .image-grid {{
                grid-template-columns:
                    repeat(2, minmax(0, 1fr));
            }}
        }}
    </style>
</head>

<body>
    <main class="page">
        <h1>
            DBSCAN Cluster Inspection
        </h1>

        <p class="subtitle">
            Variant:
            {escape(str(CLUSTERING_VARIANT_NAME))} |
            Model:
            {escape(str(MODEL_TAG))} |
            eps = {DBSCAN_EPS} |
            MinPts = {DBSCAN_MIN_SAMPLES} |
            metric = {escape(str(DBSCAN_METRIC))}
        </p>

        <p class="path-note">
            Image URL prefix:
            <code>{escape(str(IMAGE_URL_PREFIX))}</code>

            &nbsp;|&nbsp;

            Metadata column:
            <code>{escape(str(IMAGE_PATH_COLUMN))}</code>
        </p>

        <section class="summary">
            <div class="summary-card">
                <span class="summary-label">
                    Clusters
                </span>

                <span class="summary-value">
                    {number_of_report_clusters:,}
                </span>
            </div>

            <div class="summary-card">
                <span class="summary-label">
                    Clustered images
                </span>

                <span class="summary-value">
                    {number_of_clustered_images:,}
                </span>
            </div>

            <div class="summary-card">
                <span class="summary-label">
                    Images available
                </span>

                <span class="summary-value">
                    {number_of_available_images:,}
                </span>
            </div>

            <div class="summary-card">
                <span class="summary-label">
                    Noise included
                </span>

                <span class="summary-value">
                    {"Yes" if INCLUDE_NOISE else "No"}
                </span>
            </div>
        </section>

        <div class="controls">
            <button
                id="collapse-all"
                type="button"
            >
                Collapse all
            </button>

            <button
                id="unload-all"
                type="button"
            >
                Unload all images
            </button>
        </div>

        {''.join(cluster_headers)}
    </main>

    <script>
        const clusterData = {cluster_data_json};

        const clusterElements = document.querySelectorAll(
            "details.cluster"
        );

        function escapeHtml(value) {{
            return String(value)
                .replaceAll("&", "&amp;")
                .replaceAll("<", "&lt;")
                .replaceAll(">", "&gt;")
                .replaceAll('"', "&quot;")
                .replaceAll("'", "&#039;");
        }}

        function buildClusterContent(clusterElement) {{
            if (
                clusterElement.dataset.loaded === "true"
            ) {{
                return;
            }}

            const clusterPosition = Number(
                clusterElement.dataset.clusterPosition
            );

            const cluster = clusterData[
                clusterPosition
            ];

            const contentElement =
                clusterElement.querySelector(
                    ".cluster-content"
                );

            const imageCards = cluster.images
                .map(image => {{
                    let imageContent;

                    if (image.image_url) {{
                        imageContent = `
                            <a
                                class="image-link"
                                href="${{escapeHtml(image.image_url)}}"
                                target="_blank"
                                rel="noopener"
                                title="Open original crop"
                            >
                                <img
                                    src="${{escapeHtml(image.image_url)}}"
                                    alt="${{escapeHtml(image.filename)}}"
                                    loading="lazy"
                                    decoding="async"
                                    onerror="
                                        this.closest('.image-link')
                                            .outerHTML =
                                            '<div class=&quot;missing-image&quot;>Image could not be loaded</div>'
                                    "
                                >
                            </a>
                        `;
                    }} else {{
                        imageContent = `
                            <div class="missing-image">
                                No image path available
                            </div>
                        `;
                    }}

                    return `
                        <article class="image-card">
                            ${{imageContent}}

                            <div class="image-info">
                                <div class="filename">
                                    ${{escapeHtml(image.filename)}}
                                </div>

                                <div class="embedding-index">
                                    Embedding index:
                                    ${{image.embedding_index}}
                                </div>

                                <div class="stored-path">
                                    ${{escapeHtml(image.stored_path)}}
                                </div>
                            </div>
                        </article>
                    `;
                }})
                .join("");

            let truncationNote = "";

            if (cluster.hidden_count > 0) {{
                truncationNote = `
                    <p class="truncation-note">
                        ${{cluster.hidden_count.toLocaleString()}}
                        additional images are not displayed because
                        MAX_IMAGES_PER_CLUSTER is set to
                        {MAX_IMAGES_PER_CLUSTER}.
                    </p>
                `;
            }}

            contentElement.innerHTML = `
                <div class="image-grid">
                    ${{imageCards}}
                </div>

                ${{truncationNote}}
            `;

            clusterElement.dataset.loaded = "true";
        }}

        function unloadCluster(clusterElement) {{
            const contentElement =
                clusterElement.querySelector(
                    ".cluster-content"
                );

            contentElement.innerHTML = `
                <p class="loading-message">
                    Open this cluster to load its images.
                </p>
            `;

            clusterElement.dataset.loaded = "false";
        }}

        clusterElements.forEach(
            clusterElement => {{
                clusterElement.addEventListener(
                    "toggle",
                    () => {{
                        if (clusterElement.open) {{
                            buildClusterContent(
                                clusterElement
                            );
                        }}
                    }}
                );

                // Load clusters configured as initially open.
                if (clusterElement.open) {{
                    buildClusterContent(
                        clusterElement
                    );
                }}
            }}
        );

        document
            .getElementById("collapse-all")
            .addEventListener(
                "click",
                () => {{
                    clusterElements.forEach(
                        clusterElement => {{
                            clusterElement.open = false;
                        }}
                    );
                }}
            );

        document
            .getElementById("unload-all")
            .addEventListener(
                "click",
                () => {{
                    clusterElements.forEach(
                        clusterElement => {{
                            clusterElement.open = false;

                            unloadCluster(
                                clusterElement
                            );
                        }}
                    );
                }}
            );
    </script>
</body>
</html>
"""


# -----------------------------------------------------
# Save HTML report
# -----------------------------------------------------

HTML_REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

HTML_REPORT_PATH.write_text(
    html_document,
    encoding="utf-8",
)

elapsed_seconds = (
    time.perf_counter()
    - start_time
)

report_size_mb = (
    HTML_REPORT_PATH.stat().st_size
    / 1024
    / 1024
)

print("=" * 70)
print("LIGHTWEIGHT HTML CLUSTER REPORT CREATED")
print("=" * 70)
print(
    "Clusters:",
    f"{number_of_report_clusters:,}",
)
print(
    "Clustered images:",
    f"{number_of_clustered_images:,}",
)
print(
    "Images available in report:",
    f"{number_of_available_images:,}",
)
print(
    "HTML size:",
    f"{report_size_mb:.2f} MB",
)
print(
    "Runtime:",
    f"{elapsed_seconds:.2f} seconds",
)
print("Saved to:")
print(HTML_REPORT_PATH)

CREATING LIGHTWEIGHT HTML CLUSTER REPORT
HTML report: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/cluster_inspection.html
Image-path column: local_crop_path
Image URL prefix: '../../../../'
Include noise: False
Maximum images per cluster: 100
Filename label column: crop_id
Report clusters: 818
Clustered images: 18,011
Images available in report: 11,734
Prepared 100/818 clusters
Prepared 200/818 clusters
Prepared 300/818 clusters
Prepared 400/818 clusters
Prepared 500/818 clusters
Prepared 600/818 clusters
Prepared 700/818 clusters
Prepared 800/818 clusters
Prepared 818/818 clusters
LIGHTWEIGHT HTML CLUSTER REPORT CREATED
Clusters: 818
Clustered images: 18,011
Images available in report: 11,734
HTML size: 2.80 MB
Runtime: 0.66 seconds
Saved to:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/cluster_inspec

###References
- Radford et al. (2021): Learning Transferable Visual Models From Natural Language Supervision. [DOI: https://doi.org/10.48550/arXiv.2103.00020]
- Gadre et al. (2023): DataComp: In search of the next generation of multimodal datasets. [DOI: https://doi.org/10.48550/arXiv.2304.14108]